# 🧠 Anchor-Based vs. Anchor-Free Object Detection Bounding Box Decoding

Welcome to the hands-on explanation notebook for **Anchor-Based vs. Anchor-Free Decoding**! In this notebook, we will:
1. Write functions to decode raw neural network outputs into bounding box coordinates ($[x_{\min}, y_{\min}, x_{\max}, y_{\max}]$) in image space.
2. Compare the mathematics of anchor-based decoding (relies on prior anchor scales) and anchor-free decoding (relies on grid center offsets).
3. Implement both decode algorithms from scratch using NumPy.
4. Decode test cases for a specific grid cell ($5,5$) with a stride of 32.
5. Plot the decoded bounding boxes, the grid cell boundary, and centers side-by-side using Matplotlib to visualize how they differ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## 1. Mathematical Implementations in NumPy

We implement both decoding algorithms from first principles.

In [ ]:
def decode_anchor_based(grid_x, grid_y, anchor_w, anchor_h, tx, ty, tw, th, stride=32):
    # 1. Center of bbox in grid coordinates (with sigmoid adjustment)
    bx = 1.0 / (1.0 + np.exp(-tx)) + grid_x
    by = 1.0 / (1.0 + np.exp(-ty)) + grid_y
    
    # 2. Width and height in image space
    bw = anchor_w * np.exp(tw)
    bh = anchor_h * np.exp(th)
    
    # 3. Center scaled to image space
    bx_img = bx * stride
    by_img = by * stride
    
    # 4. Convert to XYXY
    x_min = bx_img - bw / 2
    y_min = by_img - bh / 2
    x_max = bx_img + bw / 2
    y_max = by_img + bh / 2
    return np.array([x_min, y_min, x_max, y_max])

def decode_anchor_free(grid_x, grid_y, l, t, r, b, stride=32):
    # 1. Center point of the cell in image space
    cx = (grid_x + 0.5) * stride
    cy = (grid_y + 0.5) * stride
    
    # 2. Coordinates in image space applying left, top, right, bottom offsets
    x_min = cx - l * stride
    y_min = cy - t * stride
    x_max = cx + r * stride
    y_max = cy + b * stride
    return np.array([x_min, y_min, x_max, y_max])

## 2. Test Case Decoding

We decode sample values for grid coordinate $(5,5)$ with a stride of 32 (mapping to image coordinate region around $160$ to $192$).

In [ ]:
stride = 32
grid_x, grid_y = 5.0, 5.0

# Anchor-based inputs
anchor_w, anchor_h = 64.0, 128.0
tx, ty, tw, th = 0.2, -0.4, 0.1, -0.2

box_anchor_based = decode_anchor_based(grid_x, grid_y, anchor_w, anchor_h, tx, ty, tw, th, stride)

# Anchor-free inputs
l, t, r, b = 1.2, 1.8, 1.5, 2.2
box_anchor_free = decode_anchor_free(grid_x, grid_y, l, t, r, b, stride)

print("Decoded Anchor-Based Box:", [round(float(coord), 2) for coord in box_anchor_based])
print("Decoded Anchor-Free Box: ", [round(float(coord), 2) for coord in box_anchor_free])

## 3. Visualization

Let's visualize the grid cell $(5,5)$ and the decoded boxes in image space.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(80, 280)
ax.set_ylim(280, 80) # Inverted Y-axis to match image space

# Draw grid cell (5,5)
cell_x1 = grid_x * stride
cell_y1 = grid_y * stride
rect_cell = patches.Rectangle((cell_x1, cell_y1), stride, stride, linewidth=2, edgecolor='black', facecolor='yellow', alpha=0.1, label='Grid Cell (5,5)')
ax.add_patch(rect_cell)
ax.plot((grid_x + 0.5)*stride, (grid_y + 0.5)*stride, 'ko', label='Cell Center')

# Draw Decoded Anchor-Based Box
w_ab = box_anchor_based[2] - box_anchor_based[0]
h_ab = box_anchor_based[3] - box_anchor_based[1]
rect_ab = patches.Rectangle((box_anchor_based[0], box_anchor_based[1]), w_ab, h_ab, linewidth=3, edgecolor='blue', facecolor='none', label='Decoded Anchor-Based Box')
ax.add_patch(rect_ab)

# Draw Decoded Anchor-Free Box
w_af = box_anchor_free[2] - box_anchor_free[0]
h_af = box_anchor_free[3] - box_anchor_free[1]
rect_af = patches.Rectangle((box_anchor_free[0], box_anchor_free[1]), w_af, h_af, linewidth=3, edgecolor='green', facecolor='none', linestyle='--', label='Decoded Anchor-Free Box')
ax.add_patch(rect_af)

ax.set_title("Anchor-Based vs. Anchor-Free Box Decoding Comparison", fontsize=14)
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()
plt.show()

## 4. Summary of Key Differences

| Feature | Anchor-Based (YOLOv5) | Anchor-Free (YOLOv8/11) |
| :--- | :--- | :--- |
| **Box Priors** | Pre-calculated K-Means anchor shapes ($p_w, p_h$). | None. Models boundary distances directly. |
| **Target Dimensions** | Logarithmic height/width ratios ($e^{t_w}$). | Linear pixel/grid distances ($l, t, r, b$). |
| **Custom Datasets** | Requires clustering recalculation for strange shapes. | Generalizes automatically to any shape. |
| **Output Channels** | High (e.g. 3 anchors $\times$ (5 + num_classes) values). | Low (e.g. 4 boundaries + num_classes values). |